In [ ]:
import os
os.chdir('f:\\Projects\\GeneIndex')

In [ ]:
from src.u1_downloader.images_downloader import download_group, get_all_gids
from src.u1_downloader.images_loader import load_all
import pandas as pd
from pathlib import Path
import torch
import torchvision
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from torchvision.transforms import v2
import torch.nn.functional as F
import h5py
import numpy as np
import shutil
import math

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset
import bisect

In [ ]:
DS_PATH = Path('src/m0_ae/dataset/')

In [ ]:
preprocess = v2.Compose([
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
    # v2.Normalize(mean=[0.485,0.456,0.406], std=[0.229, 0.224, 0.225])
])

def normalize_image(x):
    x_min = x.amin(dim=(1, 2), keepdim=True)
    x_max = x.amax(dim=(1, 2), keepdim=True)
    return (x - x_min) / (x_max - x_min + 1e-8)

def standardize_image(x):
    mean = x.mean(dim=(1, 2), keepdim=True)
    std = x.std(dim=(1, 2), keepdim=True)
    return (x - mean) / (std + 1e-8)

class MultiH5Dataset(Dataset):
    def __init__(self, h5_files, dataset_name="patches"):
        self.h5_files = h5_files
        self.dataset_name = dataset_name

        self.lengths = []
        for path in h5_files:
            with h5py.File(path, "r") as f:
                self.lengths.append(len(f[dataset_name]))

        # Prefix sums for global indexing
        self.cumulative = [0]
        for l in self.lengths:
            self.cumulative.append(self.cumulative[-1] + l)

        self._files = [
            h5py.File(path, "r") for path in self.h5_files
        ]

    def __len__(self):
        return self.cumulative[-1]

    def __getitem__(self, idx):
        file_idx = bisect.bisect_right(self.cumulative, idx) - 1
        local_idx = idx - self.cumulative[file_idx]

        x = self._files[file_idx][self.dataset_name][local_idx]

        x = torch.from_numpy(x).float()/255

        x = preprocess(x)
        # x = standardize_image(x)

        return x

    def __del__(self):
        if self._files is not None:
            for f in self._files:
                try:
                    f.close()
                except Exception:
                    pass 
ds = MultiH5Dataset(list(DS_PATH.glob('train_100k/*'))) 
print('Len', len(ds))

In [ ]:
plt.imshow(ds[300].permute(1,2,0))
plt.show() 

In [ ]:
class MAEDecoder(nn.Module):
    def __init__(self, image_size=512, patch_size=16, d_model=256):
        super(MAEDecoder, self).__init__()
        
        self.num_patches = (image_size//patch_size)**2
        self.d_model = d_model
        self.img_size = image_size
        self.patch_size = patch_size

        self.posenc = nn.Parameter(torch.randn(1, self.num_patches, d_model))

        self.blocks = nn.TransformerEncoder(nn.TransformerEncoderLayer(
            d_model, 8, 2*d_model,
            norm_first=True, 
            batch_first=True
        ), num_layers=2)

        # self.depatch = nn.ConvTranspose2d(d_model, 3, patch_size, patch_size)

        def dub(cin,cout,K,s,p):
            return nn.Sequential(
                nn.Conv2d(cin, cout, K, s, p), nn.BatchNorm2d(cout), nn.GELU(),
                nn.Conv2d(cout, cout, K, s, p), nn.BatchNorm2d(cout), nn.GELU(),
            )
        

        self.depatch = nn.Sequential( # Inp: [N, 256, 32, 32]
                nn.Upsample(scale_factor=2), 
                dub(self.d_model, 128,3,1,1), # [N, 128, 64, 64]
                nn.Upsample(scale_factor=2), 
                dub(128, 64,3,1,1), # [N, 64, 128, 128]
                nn.Upsample(scale_factor=2), 
                dub(64, 32,3,1,1), # [N, 32, 256, 256]
                nn.Upsample(scale_factor=2), 
                dub(32,16,3,1,1),
                # nn.Conv2d(32, 16,1,1,0), nn.BatchNorm2d(16), nn.LeakyReLU(),
                nn.Conv2d(16,3,1,1,0),
            )  

    def forward(self, x):
        p = x.flatten(-2).permute(0,2,1) # [N, 1024, 256]
        p_emb = p + self.posenc

        latent = self.blocks(p_emb) # [N, 1024, 256]
        latent = latent.permute(0, 2, 1).reshape(-1, self.d_model, self.img_size//self.patch_size, self.img_size//self.patch_size) 

        return self.depatch(latent)
    
class ConvDecoder(nn.Module):
    def __init__(self, image_size=512, patch_size=16, d_model=256):
        super(ConvDecoder, self).__init__()
        
        self.num_patches = (image_size//patch_size)**2
        self.d_model = d_model
        self.img_size = image_size
        self.patch_size = patch_size

        def dub(cin,cout,K,s,p):
            return nn.Sequential(
                nn.Conv2d(cin, cout, K, s, p), nn.BatchNorm2d(cout), nn.GELU(),
                nn.Conv2d(cout, cout, K, s, p), nn.BatchNorm2d(cout), nn.GELU(),
            )
        

        self.upscale = nn.Sequential( # Inp: [N, 256, 32, 32]
                nn.Upsample(scale_factor=2), 
                dub(self.d_model, 128,3,1,1), # [N, 128, 64, 64]
                nn.Upsample(scale_factor=2), 
                dub(128, 64,3,1,1), # [N, 64, 128, 128]
                nn.Upsample(scale_factor=2), 
                dub(64, 32,3,1,1), # [N, 32, 256, 256]
                nn.Upsample(scale_factor=2), 
                dub(32,16,3,1,1),
                # nn.Conv2d(32, 16,1,1,0), nn.BatchNorm2d(16), nn.LeakyReLU(),
                nn.Conv2d(16,3,1,1,0),
            )  

    def forward(self, x):
        return self.upscale(x)

In [ ]:
SCAN_WIDTH = 1920
SCAN_HEIGHT = 1600

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

class Att(nn.Module):
    def __init__(self, d_model, num_heads=8):
        super(Att, self).__init__()

        assert d_model % num_heads == 0

        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.qkv = nn.Linear(d_model, d_model*3)
        self.proj = nn.Linear(d_model, d_model)
 
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        out = F.scaled_dot_product_attention(q, k, v)
        out = out.transpose(1, 2).reshape(B, N, C)
        return self.proj(out)

def window_partition(x, window_size):
    """[B, H, W, C] -> [B * num_windows, window_size*window_size, C]"""
    B, H, W, C = x.shape
    pad_h = (window_size - H % window_size) % window_size
    pad_w = (window_size - W % window_size) % window_size
    if pad_h or pad_w:
        x = F.pad(x, (0, 0, 0, pad_w, 0, pad_h))
    Hp, Wp = H + pad_h, W + pad_w
    x = x.view(B, Hp // window_size, window_size, Wp // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).reshape(-1, window_size * window_size, C)
    return windows, (Hp, Wp)
 
 
def window_unpartition(windows, window_size, pad_hw, hw):
    """Reverse of window_partition."""
    Hp, Wp = pad_hw
    H, W = hw
    B = windows.shape[0] // ((Hp // window_size) * (Wp // window_size))
    C = windows.shape[-1]
    x = windows.view(B, Hp // window_size, Wp // window_size, window_size, window_size, C)
    x = x.permute(0, 1, 3, 2, 4, 5).reshape(B, Hp, Wp, C)
    if Hp > H or Wp > W:
        x = x[:, :H, :W, :].contiguous()
    return x
 
 
class Block(nn.Module):

    def __init__(self, d_model, num_heads=8, window_size=0):
        super().__init__()
        self.window_size = window_size
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = Att(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, d_model*4),
            nn.GELU(),
            nn.Linear(d_model*4, d_model),
        )
 
    def forward(self, x, H, W):
        B, N, C = x.shape
        use_window = self.window_size > 0 and N == H * W
 
        shortcut = x
        x = self.ln1(x)
 
        if use_window:
            x = x.view(B, H, W, C)
            x, pad_hw = window_partition(x, self.window_size)
 
        x = self.attn(x)
 
        if use_window:
            x = window_unpartition(x, self.window_size, pad_hw, (H, W))
            x = x.reshape(B, N, C)
 
        x = shortcut + x
        x = x + self.mlp(self.ln2(x))
        return x


In [ ]:
class MAEEncoder(nn.Module):
    def __init__(self, image_size=512, patch_size=16, d_model=256):
        super(MAEEncoder, self).__init__()
        
        self.num_patches = (image_size//patch_size)**2
        self.d_model = d_model
        self.img_size = image_size
        self.patch_size = patch_size

        self.patch_proj = nn.Conv2d(3, d_model, patch_size, patch_size)

        self.posenc = nn.Parameter(torch.randn(1, d_model, image_size//patch_size, image_size//patch_size))
        self.posenc_interp = nn.Parameter(torch.randn(1, d_model, SCAN_HEIGHT//patch_size, SCAN_WIDTH//patch_size))

        self.blocks = nn.ModuleList([
            Block(d_model, 8, 16),
            Block(d_model, 8, 16),
            Block(d_model, 8, 16),
            Block(d_model, 8, 0),

            Block(d_model, 8, 16),
            Block(d_model, 8, 16),
            Block(d_model, 8, 16),
            Block(d_model, 8, 0),
            
            Block(d_model, 8, 16),
            Block(d_model, 8, 16),
            Block(d_model, 8, 16),
            Block(d_model, 8, 0),
        ])

        self.mask_token = nn.Parameter(torch.randn(1, 1, d_model))
    
    def update_posec_interp(self):
        with torch.no_grad():
            self.posenc_interp.copy_(torch.nn.functional.interpolate(self.posenc.detach(), 
                                                                    size=(SCAN_HEIGHT//self.patch_size, 
                                                                        SCAN_WIDTH//self.patch_size), 
                                                                    mode="bilinear"))


    def forward(self, x, mask_perc=0.7):
        if not self.training:
            mask_perc = 0

        p = self.patch_proj(x) # [N, d_model, x, y]

        pnum_x = p.size(-2)
        pnum_y = p.size(-1)

        seqlen = pnum_x * pnum_y
        
        if x.size(-1) == 512:
            p_emb = p + self.posenc
        else:
            assert x.size(-1) == 1920, 'scan must me of size 1920x1600'
            p_emb = p + self.posenc_interp
                                           
        p_emb = p_emb.flatten(-2).permute(0,2,1) # [N, 1024, d_model]


        rand = torch.rand(x.size(0), seqlen, device=x.device)
        shuffle_idx = rand.argsort(dim=1)
        idx_keep = shuffle_idx[:, 0:math.ceil((1-mask_perc)*seqlen)]
        idx_restore = shuffle_idx.argsort(dim=1)

        p_emb = torch.gather(
            p_emb,
            dim=1,
            index=idx_keep.unsqueeze(-1).expand(-1, -1, p_emb.size(-1))
        )


        for b in self.blocks:
            p_emb = b(p_emb, pnum_x, pnum_y) # [N, seqlen, d_model]
        latent = p_emb

        latent = torch.cat([latent, 
                            self.mask_token.repeat(x.size(0), seqlen-latent.size(1), 1)],
                            dim=1)

        lat_rest = torch.gather(
            latent,
            dim=1,
            index=idx_restore.unsqueeze(-1).expand(-1, -1, latent.size(-1))
        )
        lat_rest = lat_rest.permute(0, 2, 1).reshape(-1, self.d_model, pnum_x, pnum_y) # [N, 256, 32, 32]
        return lat_rest # , idx_keep.reshape()

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEVICE

In [ ]:
torch.cuda.empty_cache()
torch.cuda.synchronize()

In [ ]:
enc = MAEEncoder(d_model=384, patch_size=16).to(DEVICE)
dec = ConvDecoder(d_model=384, patch_size=16).to(DEVICE)
print('Enc', sum([p.numel() for p in enc.parameters()]))
print('Dec', sum([p.numel() for p in dec.parameters()]))



In [ ]:
dl = torch.utils.data.DataLoader(ds, batch_size=8, shuffle=True) # 16 is MAXXX

opt = torch.optim.AdamW(list(enc.parameters()) + list(dec.parameters()), lr=0.001, weight_decay=0.05)
crit = nn.MSELoss()

for epoch in range(50):
    t = tqdm(dl, desc=f'Epoch {epoch+1}')
    t_nt = 0
    enc.train()
    dec.train()
    for z, x in enumerate(t):
        # plt.imshow(x[0].permute(1,2,0))
        # plt.show()
        x = x.to(DEVICE)

        lat = enc(x, mask_perc=0.75)
        pred = dec(lat)
        # pred = normalize_patches(pred)

        loss = crit(x, pred)

        loss.backward()
        opt.step()
        opt.zero_grad()

        t.set_postfix_str(f'Loss: {loss.item():.3f} TvsNT: {t_nt}')

        if z%100 == 0:
            plt.imshow(x.cpu()[0].permute(1,2,0))
            plt.show()
            plt.imshow(pred.detach().cpu()[0].permute(1,2,0))
            plt.show()

            t_nt = score_model(enc, verbose=True)


In [ ]:
import pickle as pkl
pkl.dump(enc, Path('src/m0_ae/enc_7e.pkl').open('wb'))

# Metrics

In [ ]:
from sklearn.decomposition import PCA


def extract_features(encoder, img):
    # Img: [3, w, h]

    C, W, H = img.shape
    # img = img.resize((W//4, H//4))
    # W = W//4
    # H = H//4

    x = math.ceil(W / 512)
    y = math.ceil(H / 512)

    pad_w = x * 512 - W
    pad_h = y * 512 - H

    img = F.pad(img, (0, pad_h, 0, pad_w))
    patches = (
        img.unfold(1, 512, 512)
        .unfold(2, 512, 512)
        .permute(1, 2, 0, 3, 4)
        .contiguous()
    ) # f.e. [7, 8, 3, 512, 512]

    batch = patches.reshape(-1,3,512,512)
    batch = preprocess(batch)
    # batch = torch.stack([standardize_image(i) for i in batch])
    # mean = torch.mean(batch, dim=(0,1,2,3))
    # std = torch.std(batch, dim=(0,1,2,3))
    
    # batch = (batch - mean)/(std+1e-8)

    plt.imshow(batch[1].permute(1,2,0))
    plt.show()
    # assert False

    encoder.eval()
    with torch.no_grad():
        feats = encoder(batch.to(DEVICE), mask_perc=0).detach().cpu() # [x*y, C, 32, 32]
        Fdim, FW, FH = feats.size(1), feats.size(2), feats.size(3) 
        patch_size = 512//FW
        # print(patch_size)
        
    encoder.train()

    features_unflatten = feats.reshape(x,y,Fdim,FW,FH)

    features_image = features_unflatten.permute(2, 0, 3, 1, 4).reshape(Fdim, x * FW, y * FH)

    features_image_crop = features_image[:, :W//patch_size, :H//patch_size]
    # print(features_image.size())

    return features_image_crop
# extract_features(enc, img_to_extract)

In [ ]:
def score_model(encoder, verbose=False):
    img_to_extract = Image.open('src/m0_ae/dataset/scoring_ds/scoring2.jpg')
    img_to_extract = img_to_extract.resize((SCAN_WIDTH, SCAN_HEIGHT))

    img_to_extract = torchvision.transforms.functional.to_tensor(img_to_extract)
    # print(img_to_extract.size())
    encoder.eval()
    encoder.update_posec_interp()
    with torch.no_grad():
        # %timeit encoder(img_to_extract.unsqueeze(0).to(DEVICE)).squeeze(0).detach().cpu()
        feats = encoder(img_to_extract.unsqueeze(0).to(DEVICE)).squeeze(0).detach().cpu()
    encoder.train()
    # assert False
    Fdim, W, H = feats.shape
    feats_flat = feats.permute(1,2,0).reshape(-1, Fdim) # [G, 384], G is big
    
    # print(feats_flat.size())

    # PCA
    # pca = PCA(n_components=1)
    # out_feats_flat = pca.fit_transform(feats_flat) # [1024]
    # out_feats = out_feats_flat.reshape(feats.size(1), feats.size(2))

    # out_feats = out_feats - out_feats.min()
    # out_feats = out_feats / out_feats.max()
    # plt.matshow(out_feats,cmap='viridis')
    # plt.show()


    # Text vs not text
    t_nt = 0
    t_nt += F.cosine_similarity(feats[:, 66:100, 20:80].mean(dim=(-1,-2)), feats[:, 20:40, 20:80].mean(dim=(-1,-2)), dim=0)
    # t_nt += F.cosine_similarity(feats[1, :, 1:3, 7:13].mean(dim=(-1,-2)), feats[1, :, 17:22, 8:12].mean(dim=(-1,-2)), dim=0)

    csim = F.cosine_similarity(feats[:, 66:100, 20:80].mean(dim=(-1,-2)).unsqueeze(0), feats_flat, dim=1).reshape(W,H)

    if verbose:
        fig, ax = plt.subplots(1,2, figsize=(15,15))
        ax[0].imshow(img_to_extract.permute(1,2,0))
        ax[1].matshow(csim)
        plt.show()
    print(f'{t_nt=}')

    return t_nt
score_model(enc, verbose=True)
